# Sales Crediting Platform Prototype (with AI-Powered Root Cause Analysis)

Built by **Rutuja More** as a portfolio project demonstrating the stack a pharma
**Sales Crediting / Incentive Compensation Data Analyst / Data Engineer** role needs:
SQL/Python data engineering, business-rule translation, reconciliation frameworks,
anomaly detection, and a GenAI layer for root-cause analysis and stakeholder reporting.

**Click `Runtime > Run all` above to execute the entire pipeline in this browser — no setup required.**

Full code + README: *(add your GitHub repo link here)*

---
### Core design principle
Python computes and verifies every number; the LLM only narrates and hypothesizes
**root causes** in plain English for non-technical Incentive Compensation stakeholders.
The model is explicitly told not to invent numbers — only to reason over facts it's given.

---
## Step 1 — Install dependencies


In [ ]:
!pip install -q pandas anthropic


## Step 2 — Generate mock data
Creates a realistic dataset: 40 reps, 25 territories (some with shared/split
credit rules), 4,000 raw sales transactions across 3 source systems — with one
intentionally broken credit rule injected to demonstrate the reconciliation
framework catching it.

In [ ]:
"""
generate_mock_data.py
----------------------
Creates a realistic mock pharma Sales Crediting / Incentive Compensation
dataset: raw sales transactions from multiple source systems, territory
alignments, reps, and credit-split business rules.

This mirrors what a Sales Crediting Platform ingests in real life:
- Sales data lands from multiple sources (specialty pharmacy, IQVIA-style
  claims data, direct orders) and needs to be reconciled and credited to
  the right sales reps according to territory alignment rules.
- Some territories are "shared" (e.g., a key account split between a
  primary and secondary rep) which is where crediting gets non-trivial
  and is a common source of real-world data quality issues.

Run: python generate_mock_data.py
Outputs: reps.csv, territories.csv, credit_rules.csv, raw_sales.csv
"""

import csv
import random
from datetime import date, timedelta

random.seed(7)

REGIONS = ["Northeast", "Southeast", "Midwest", "West"]
PRODUCTS = ["OncoPlex", "ImmunoGuard", "CardioFlex", "NeuroCare"]
SOURCE_SYSTEMS = ["SpecialtyPharmacy", "IQVIA_Claims", "DirectOrder"]

# ---- Reps ----
reps = []
for i in range(1, 41):
    reps.append({"rep_id": f"REP{i:03d}", "rep_name": f"Rep {i}",
                 "region": random.choice(REGIONS)})

# ---- Territories ----
territories = []
for i in range(1, 26):
    territories.append({"territory_id": f"TERR{i:03d}",
                         "region": random.choice(REGIONS)})

# ---- Credit rules: most territories map 1 rep = 100%; some are shared ----
credit_rules = []
for t in territories:
    primary_rep = random.choice(reps)
    if random.random() < 0.2:
        # shared territory: primary/secondary split
        secondary_rep = random.choice([r for r in reps if r["rep_id"] != primary_rep["rep_id"]])
        split = random.choice([70, 60, 50])
        credit_rules.append({"territory_id": t["territory_id"], "rep_id": primary_rep["rep_id"], "credit_pct": split})
        credit_rules.append({"territory_id": t["territory_id"], "rep_id": secondary_rep["rep_id"], "credit_pct": 100 - split})
    else:
        credit_rules.append({"territory_id": t["territory_id"], "rep_id": primary_rep["rep_id"], "credit_pct": 100})

# Intentionally inject a handful of BROKEN rules (splits that don't sum to 100)
# -- this is exactly the kind of data quality issue a reconciliation framework should catch.
broken_territory = random.choice(territories)["territory_id"]
for rule in credit_rules:
    if rule["territory_id"] == broken_territory:
        rule["credit_pct"] = round(rule["credit_pct"] * 0.8)  # now sums to <100%

# ---- Raw sales transactions ----
start_date = date(2026, 1, 1)
rows = []
txn_id = 5000
for i in range(4000):
    territory = random.choice(territories)
    product = random.choice(PRODUCTS)
    source = random.choice(SOURCE_SYSTEMS)
    units = random.randint(1, 50)
    unit_price = {"OncoPlex": 1200, "ImmunoGuard": 850, "CardioFlex": 400, "NeuroCare": 600}[product]
    amount = round(units * unit_price * random.uniform(0.9, 1.1), 2)

    # inject occasional duplicate-looking / spike transactions for anomaly detection
    if random.random() < 0.02:
        amount *= random.uniform(3, 5)

    txn_date = start_date + timedelta(days=random.randint(0, 250))
    rows.append({
        "transaction_id": f"TXN{txn_id}",
        "date": txn_date.isoformat(),
        "territory_id": territory["territory_id"],
        "product": product,
        "source_system": source,
        "units": units,
        "sales_amount": amount,
    })
    txn_id += 1

rows.sort(key=lambda r: r["date"])

with open("reps.csv", "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=reps[0].keys()); w.writeheader(); w.writerows(reps)

with open("territories.csv", "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=territories[0].keys()); w.writeheader(); w.writerows(territories)

with open("credit_rules.csv", "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=credit_rules[0].keys()); w.writeheader(); w.writerows(credit_rules)

with open("raw_sales.csv", "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=rows[0].keys()); w.writeheader(); w.writerows(rows)

print(f"Generated {len(reps)} reps, {len(territories)} territories, "
      f"{len(credit_rules)} credit rules, {len(rows)} raw sales transactions.")
print(f"NOTE: territory {broken_territory} has an intentionally broken credit "
      f"split (<100%) to demo the reconciliation framework catching it.")


## Step 3 — ETL pipeline
Ingest → validate credit rules (data quality gate) → apply credit-split logic →
reconcile credited totals against source → summarize by rep/month.

In [ ]:
"""
etl_pipeline.py
-----------------
The core Sales Crediting Platform logic: ingests raw sales transactions
from multiple source systems and applies territory-based credit-split
business rules to produce a credited-sales output table.

Written in pandas for portability/demo purposes, but the transformations
map 1:1 onto PySpark/Databricks (see README "Porting to PySpark" section)
-- every operation here (read_csv, merge, groupby, filter) has a direct
PySpark equivalent (spark.read.csv, join, groupBy, filter).

PIPELINE STAGES (mirrors a real ETL/ELT job):
  1. INGEST   -- load raw sales + reference tables (reps, territories, rules)
  2. VALIDATE -- check credit rules sum to 100% per territory (data quality gate)
  3. TRANSFORM-- explode each sale into per-rep credited rows using the
                 credit-split rules ("who gets credit for this sale, and
                 how much")
  4. LOAD     -- write the credited-sales output table
"""

import pandas as pd


def ingest(sales_path="raw_sales.csv", rules_path="credit_rules.csv",
           reps_path="reps.csv", territories_path="territories.csv"):
    sales = pd.read_csv(sales_path, parse_dates=["date"])
    rules = pd.read_csv(rules_path)
    reps = pd.read_csv(reps_path)
    territories = pd.read_csv(territories_path)
    return sales, rules, reps, territories


def validate_credit_rules(rules: pd.DataFrame) -> pd.DataFrame:
    """
    Data quality gate: every territory's credit_pct values must sum to 100.
    Returns a DataFrame of BROKEN territories (this is exactly the kind of
    issue a real Sales Crediting Platform must catch before it silently
    over/under-credits reps -- a direct P&L / compensation accuracy risk).
    """
    totals = rules.groupby("territory_id")["credit_pct"].sum().reset_index()
    totals.columns = ["territory_id", "total_pct"]
    broken = totals[totals["total_pct"] != 100].copy()
    return broken


def apply_credit_split(sales: pd.DataFrame, rules: pd.DataFrame) -> pd.DataFrame:
    """
    Explodes each sales transaction into one row per rep entitled to credit
    for that territory, per the credit-split rules. This is the central
    business-rule translation the JD describes: "translate sales crediting
    rules into technical requirements and implement scalable solutions."
    """
    merged = sales.merge(rules, on="territory_id", how="left")
    merged["credited_amount"] = merged["sales_amount"] * (merged["credit_pct"] / 100.0)
    merged["month"] = merged["date"].dt.to_period("M").astype(str)
    return merged


def reconcile(credited: pd.DataFrame, sales: pd.DataFrame) -> pd.DataFrame:
    """
    Reconciliation check: for each transaction, total credited_amount across
    all reps should equal the original sales_amount (i.e., 100% of every
    sale gets credited to someone, no more, no less). Any transaction that
    fails this check indicates a broken credit rule and needs escalation
    -- this is the "data quality, validation and reconciliation framework"
    the JD calls for directly.
    """
    txn_totals = credited.groupby("transaction_id")["credited_amount"].sum().reset_index()
    txn_totals.columns = ["transaction_id", "total_credited"]
    check = sales[["transaction_id", "sales_amount"]].merge(txn_totals, on="transaction_id")
    check["variance"] = (check["total_credited"] - check["sales_amount"]).round(2)
    breaks = check[check["variance"].abs() > 0.01]
    return breaks


def rep_month_summary(credited: pd.DataFrame) -> pd.DataFrame:
    """Aggregated output: credited sales by rep by month -- what an Incentive
    Compensation cycle actually consumes to calculate payouts."""
    summary = credited.groupby(["rep_id", "month"])["credited_amount"].sum().reset_index()
    return summary.sort_values(["rep_id", "month"])


def run_pipeline():
    sales, rules, reps, territories = ingest()

    broken_rules = validate_credit_rules(rules)
    credited = apply_credit_split(sales, rules)
    reconciliation_breaks = reconcile(credited, sales)
    summary = rep_month_summary(credited)

    credited.to_csv("credited_sales_output.csv", index=False)
    summary.to_csv("rep_month_summary.csv", index=False)

    return {
        "broken_credit_rules": broken_rules,
        "reconciliation_breaks": reconciliation_breaks,
        "rep_month_summary": summary,
        "total_transactions": len(sales),
        "total_credited_rows": len(credited),
    }


## Step 4 — Anomaly detection
Rep-level payout spikes and source-system transaction-volume anomalies,
month over month.

In [ ]:
"""
anomaly_detection.py
----------------------
Root-cause-style anomaly detection on top of the credited sales output.
Covers the JD's "monitor platform performance, identify data/process
issues, perform root-cause analysis" requirement.

Two kinds of anomalies detected:
  1. Rep-level payout spikes -- a rep's month-over-month credited amount
     jumps unusually (could be legitimate, could be a broken credit rule
     or duplicate transaction feed -- needs investigation either way).
  2. Source-system spikes -- a source system (e.g., IQVIA_Claims) suddenly
     sends a much larger volume of transactions than usual, which often
     indicates an upstream feed issue (duplicate file load, late-arriving
     backdated data, etc.) rather than a real sales trend.
"""

import pandas as pd


def rep_payout_anomalies(rep_month_summary: pd.DataFrame, threshold=0.5) -> pd.DataFrame:
    """Flags rep-months where credited_amount moved >= threshold (50% default)
    vs. the prior month for that rep."""
    df = rep_month_summary.sort_values(["rep_id", "month"]).copy()
    df["prev_amount"] = df.groupby("rep_id")["credited_amount"].shift(1)
    df["pct_change"] = (df["credited_amount"] - df["prev_amount"]) / df["prev_amount"]
    anomalies = df[df["pct_change"].abs() >= threshold].dropna(subset=["pct_change"])
    return anomalies[["rep_id", "month", "prev_amount", "credited_amount", "pct_change"]]


def source_system_anomalies(sales: pd.DataFrame, threshold=0.4) -> pd.DataFrame:
    """Flags source-system/month combinations with an unusual transaction
    count spike vs. the prior month -- a common early signal of an upstream
    feed problem (duplicate loads, delayed files landing all at once)."""
    sales = sales.copy()
    sales["month"] = pd.to_datetime(sales["date"]).dt.to_period("M").astype(str)
    counts = sales.groupby(["source_system", "month"]).size().reset_index(name="txn_count")
    counts = counts.sort_values(["source_system", "month"])
    counts["prev_count"] = counts.groupby("source_system")["txn_count"].shift(1)
    counts["pct_change"] = (counts["txn_count"] - counts["prev_count"]) / counts["prev_count"]
    anomalies = counts[counts["pct_change"].abs() >= threshold].dropna(subset=["pct_change"])
    return anomalies


## Step 5 — AI layer (chained prompts)
Step A: root-cause hypothesis for each data quality issue found.
Step B: leadership-ready operational readiness summary for an IC cycle kickoff.

Set `ANTHROPIC_API_KEY` below to get real LLM output — otherwise this runs in
**dry-run mode**: every calculation is real, and you'll see the exact prompts
that would be sent.

In [ ]:
import os
# Uncomment and paste a key to get real AI-generated narrative output:
# os.environ["ANTHROPIC_API_KEY"] = "sk-..."


In [ ]:
"""
ai_layer.py
------------
GenAI layer covering the JD's "Technology, Automation & AI" section:
"Explore practical applications of AI for areas such as data validation,
anomaly detection, issue identification, process automation, documentation
and operational insights."

Same core design principle as before: Python computes and verifies every
number; the LLM only narrates and hypothesizes ROOT CAUSES in plain
English for non-technical Incentive Compensation stakeholders. The model
is explicitly told not to invent numbers -- only to reason over facts it's
given.

Two chained prompts:
  1. ROOT_CAUSE_PROMPT      -- given a reconciliation break or anomaly,
                                hypothesize likely causes a data engineer
                                would investigate first
  2. OPS_SUMMARY_PROMPT     -- combines all findings into a single
                                operational readiness summary for an
                                Incentive Compensation cycle kickoff
"""

import json
import os

ROOT_CAUSE_PROMPT = """You are a data engineer supporting a pharma Sales
Crediting Platform for Incentive Compensation. Below are FACTS about a
data quality issue, already calculated from the pipeline -- do not invent
or recalculate any numbers.

ISSUE FACTS (JSON):
{issue_facts}

Task: Write a short root-cause hypothesis (2-3 sentences) a data engineer
would investigate first. Consider common causes such as: a credit-split
rule that doesn't sum to 100%, a duplicate or late-arriving file from a
source system, or a territory realignment that wasn't reflected in the
rules table yet. Be specific to the facts given -- don't list generic
possibilities that don't fit the numbers shown."""


OPS_SUMMARY_PROMPT = """You are preparing an operational readiness summary
for an Incentive Compensation cycle kickoff meeting. Audience: IC
managers and Commercial Ops leadership -- non-technical, time-constrained.

PIPELINE FACTS (JSON):
{pipeline_facts}

ROOT CAUSE HYPOTHESES ALREADY GENERATED:
{root_causes}

Task: Write a single operational readiness paragraph (5-6 sentences) that:
- States whether the platform is ready to proceed with this cycle's
  crediting run, or needs issues resolved first
- Summarizes the data quality issues found, referencing the root causes
- Ends with a clear, specific recommended action and owner (e.g.,
  "IC Ops should confirm the TERR011 split correction before rerunning")

Tone: direct and operational, like a status update to leadership before
a compensation cycle -- no hedging, no generic filler."""


def call_llm(prompt, label):
    dry_run = os.environ.get("ANTHROPIC_API_KEY") is None
    if dry_run:
        print(f"\n{'=' * 70}\n[DRY RUN] Prompt -- {label}\n{'=' * 70}\n{prompt}")
        return f"[DRY RUN -- set ANTHROPIC_API_KEY for a real {label}]"
    import anthropic
    client = anthropic.Anthropic()
    resp = client.messages.create(
        model="claude-sonnet-4-6", max_tokens=400,
        messages=[{"role": "user", "content": prompt}],
    )
    return "".join(b.text for b in resp.content if b.type == "text")


def generate_root_cause(issue_facts: dict) -> str:
    prompt = ROOT_CAUSE_PROMPT.format(issue_facts=json.dumps(issue_facts, indent=2))
    return call_llm(prompt, "Root Cause Hypothesis")


def generate_ops_summary(pipeline_facts: dict, root_causes: list) -> str:
    prompt = OPS_SUMMARY_PROMPT.format(
        pipeline_facts=json.dumps(pipeline_facts, indent=2),
        root_causes="\n".join(f"- {rc}" for rc in root_causes),
    )
    return call_llm(prompt, "Operational Readiness Summary")


## Step 6 — Run the full pipeline end-to-end
(imports from the earlier cells are dropped here since everything already
shares one notebook namespace — in the standalone repo, `main.py` imports
these normally from separate files)

In [ ]:
"""
main.py
--------
Runs the full Sales Crediting Platform prototype end-to-end:
  1. ETL pipeline (ingest -> validate -> apply credit-split rules -> load)
  2. Anomaly detection (rep payout spikes, source-system volume spikes)
  3. AI layer (root-cause hypotheses + operational readiness summary)

Run:
  python generate_mock_data.py   # once, to create the mock dataset
  python main.py

Works without an API key (DRY RUN mode) -- all pipeline logic and
anomaly detection are 100% real; only the LLM narrative step is stubbed.
"""

import json
import os



def main():
    dry_run = os.environ.get("ANTHROPIC_API_KEY") is None
    if dry_run:
        print("NOTE: ANTHROPIC_API_KEY not set -- running in DRY RUN mode.")
        print("All pipeline logic and anomaly detection are real; only the "
              "LLM narrative step is stubbed.\n")

    # ---- Stage 1: ETL Pipeline ----
    sales, rules, reps, territories = ingest()
    broken_rules = validate_credit_rules(rules)
    credited = apply_credit_split(sales, rules)
    recon_breaks = reconcile(credited, sales)
    summary = rep_month_summary(credited)

    print(f"Pipeline: {len(sales)} raw transactions -> {len(credited)} credited rows")
    print(f"Data quality gate: {len(broken_rules)} territory(ies) with broken credit rules")
    print(f"Reconciliation: {len(recon_breaks)} transactions with credited != sales_amount\n")

    # ---- Stage 2: Anomaly Detection ----
    rep_anomalies = rep_payout_anomalies(summary)
    source_anomalies = source_system_anomalies(sales)
    print(f"Anomaly detection: {len(rep_anomalies)} rep payout anomalies, "
          f"{len(source_anomalies)} source-system volume anomalies\n")

    # ---- Stage 3: AI layer -- root cause + ops summary ----
    root_causes = []
    if not broken_rules.empty:
        for _, row in broken_rules.iterrows():
            issue = {
                "issue_type": "broken_credit_rule",
                "territory_id": row["territory_id"],
                "total_credit_pct": row["total_pct"],
                "expected_pct": 100,
            }
            rc = generate_root_cause(issue)
            root_causes.append(rc)

    pipeline_facts = {
        "total_transactions": len(sales),
        "broken_credit_rules_count": len(broken_rules),
        "reconciliation_breaks_count": len(recon_breaks),
        "rep_payout_anomalies_count": len(rep_anomalies),
        "source_system_anomalies_count": len(source_anomalies),
    }
    ops_summary = generate_ops_summary(pipeline_facts, root_causes)

    print("\n" + "=" * 70)
    print("FINAL OUTPUT")
    print("=" * 70)
    print(f"\nPipeline facts:\n{json.dumps(pipeline_facts, indent=2)}")
    print(f"\n--- Root Cause Hypotheses ---")
    for rc in root_causes:
        print(f"- {rc}")
    print(f"\n--- Operational Readiness Summary ---\n{ops_summary}")


if __name__ == "__main__":
    main()


---
## How this maps to the JD

| JD requirement | What's built |
|---|---|
| Own sales crediting operations, ensure accurate processing | `etl_pipeline.py` — full ingest-to-credited-output pipeline |
| Translate business rules into technical requirements | `apply_credit_split()` — turns credit-split rules (incl. shared territories) into row-level logic |
| Data quality, validation and reconciliation frameworks | `validate_credit_rules()` + `reconcile()` |
| Root-cause analysis on data/process issues | `anomaly_detection.py` — rep payout + source-system spikes |
| Practical AI/GenAI applications | `ai_layer.py` — GenAI root-cause hypotheses + ops summary |
| Communicate technical concepts to non-technical stakeholders | AI layer output is written for IC/Commercial Ops, not engineers |

### Porting to PySpark/Databricks (the JD's actual stack)
Built in pandas for portability in this notebook — every transformation maps
directly: `pd.read_csv` → `spark.read.csv`, `df.merge` → `df.join`,
`df.groupby` → `df.groupBy`, `df.to_csv` → write to a Delta table.